
# GRAHSP torus: empirical log-Gaussian vs Mor & Netzer 2012 templates

GRAHSP ships two torus prescriptions. The default is an **empirical
log-Gaussian** cool+hot dust continuum (``activategtorus``). The alternative
is the **Mor & Netzer 2012 template** torus (``activatetorus``), which
interpolates between mean / 25th / 75th-percentile observed AGN mid-IR SEDs
via ``agn_grahsp_tor_temp`` and applies a short-wavelength Gaussian cutoff at
``agn_grahsp_tor_cutoff_um``.

This example overlays the two at fixed ``l5100`` and ``fcov``, then sweeps the
MN12 temperature blend lo → avg → hi. (The two prescriptions carry different
internal 12 μm normalizations — faithfully reproduced from upstream — so
``fcov`` has a different effective scale between them; here each curve is shown
in its native normalization.)


In [ ]:
import os

os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2"

import warnings

import jax.numpy as jnp
import matplotlib.pyplot as plt
import numpy as np

from tengri.agn import compute_grahsp_sed
from tengri.analysis.plotting import setup_style

setup_style()
warnings.filterwarnings("ignore", message=".*BakedInBackend.*")

# Mid-IR window where the torus dominates (1-60 um).
wave_aa = jnp.logspace(np.log10(1e4), np.log10(6e5), 1400)
wave_um = np.asarray(wave_aa) / 1e4


def torus_only(**kw):
    """AGN SED with disc/lines/FeII suppressed so the torus stands alone."""
    return np.asarray(
        compute_grahsp_sed(
            wave_aa,
            agn_log_lbol=45.0,
            agn_grahsp_a_lines=0.0,
            agn_grahsp_a_feii=0.0,
            agn_grahsp_fcov=0.4,
            **kw,
        )
    )


fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10.5, 4.4), sharex=True)

# These torus templates carry an arbitrary "native" normalization; only the
# relative shapes are physical. Normalize both panels by the same reference
# (the default log-Gaussian peak) so the y-axis reads O(1) instead of ~1e66.
gaussian = wave_um * torus_only(torus_model="gaussian")
norm = gaussian.max()

# Panel 1: the two torus prescriptions.
ax1.plot(wave_um, gaussian / norm, lw=1.9, label="log-Gaussian (default)")
ax1.plot(
    wave_um,
    wave_um * torus_only(torus_model="mn12", agn_grahsp_tor_temp=0.0) / norm,
    lw=1.9,
    label="Mor & Netzer 2012",
)
ax1.set_title("Torus prescription")
ax1.legend(frameon=False, fontsize=9)
ax1.set_ylabel(r"$\lambda L_\lambda$ [normalized]")

# Panel 2: MN12 temperature blend.
temps = [(-1.0, "lo (25th pct)"), (0.0, "avg (mean)"), (1.0, "hi (75th pct)")]
colors = plt.cm.plasma(np.linspace(0.15, 0.8, len(temps)))
for (t, lab), c in zip(temps, colors):
    ax2.plot(
        wave_um,
        wave_um * torus_only(torus_model="mn12", agn_grahsp_tor_temp=t) / norm,
        color=c,
        lw=1.8,
        label=rf"$T_{{\rm tor}}={t:+.0f}$ — {lab}",
    )
ax2.set_title("MN12 template temperature blend")
ax2.legend(frameon=False, fontsize=8)

for ax in (ax1, ax2):
    ax.set_xscale("log")
    ax.set_xlabel(r"rest wavelength [$\mu$m]")

fig.tight_layout()
plt.show()